Read SPRUCE data


In [ ]:
import pandas as pd
import glob
import os

base_dir = "../../data/spruce_data/WEW_Complete_Environ_20240328"

file_paths = glob.glob(os.path.join(base_dir, "*Complete_Environ_20240328.csv"))
# print(file_paths)
dfs = []
for file_path in file_paths:
    df = pd.read_csv(file_path, comment='#')
    df = df[['TIMESTAMP', 'TA_2_0__1', 'Plot', 'Temp_target', 'CO2_trmt']]
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
# print(data)
data['timestamp'] = pd.to_datetime(data['TIMESTAMP'].str.strip(), errors='coerce')

mask = data['timestamp'].isna()
if mask.any():
    data.loc[mask, 'timestamp_str'] = data.loc[mask, 'TIMESTAMP'].str.strip()
    data.loc[mask, 'timestamp_extracted'] = data.loc[mask, 'timestamp_str'].str.extract(
        r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', expand=False
    )
    data.loc[mask, 'timestamp'] = pd.to_datetime(data.loc[mask, 'timestamp_extracted'], errors='coerce')

data['year'] = data['timestamp'].dt.year
# print(data['year'])
# print(data['timestamp'])
data_clean = data.dropna(subset=['timestamp', 'TA_2_0__1'])

df_annual = data_clean.groupby(['year', 'Plot'])['TA_2_0__1'].mean().reset_index()
df_annual.rename(columns={'TA_2_0__1': 'annual_temp'}, inplace=True)
# print(df_annual)
pivot_df = df_annual.pivot(index="Plot", columns="year", values="annual_temp")

pivot_df.columns = [f"annual_temp_{int(year)}" for year in pivot_df.columns]

df_envir = pivot_df.reset_index()
df_envir.rename(columns={'Plot': 'plot'}, inplace=True)

# print(df_envir.columns)


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/spruce_data/ORNL_PhenoCam/Phenology_TransitionDates_3day_2015_2023.csv")

sos_df = df[df['direction'] == 'rising'].pivot_table(
    index=['veg_type', 'plot'], 
    columns='year', 
    values='transition_doy'
)
sos_df.columns = ['sos_' + str(col) for col in sos_df.columns]
sos_df = sos_df.reset_index()

eos_df = df[df['direction'] == 'falling'].pivot_table(
    index=['veg_type', 'plot'], 
    columns='year', 
    values='transition_doy'
)
eos_df.columns = ['eos_' + str(col) for col in eos_df.columns]
eos_df = eos_df.reset_index()

result_df = pd.merge(sos_df, eos_df, on=['veg_type', 'plot'], how='outer')

df = result_df.merge(df_envir, on="plot", how="inner")
# df.to_csv("")

# print(result_df)
# veg_type = "SH"
# df_pheno = result_df[result_df["veg_type"] == veg_type]

df_pheno = result_df

df_pheno = df_pheno.merge(df_envir, on="plot", how="inner")
elevated_plots = [4, 10, 11, 16, 19]
df_pheno['co2_level'] = np.where(df_pheno['plot'].isin(elevated_plots), 'elevated', 'ambient')
df_pheno = df_pheno[df_pheno["co2_level"] == "ambient"]
# print(df_pheno.shape)
# print(df_pheno.head())
# print(15*9)
eos_cols = df_pheno.loc[:, df_pheno.columns.str.contains('eos')]
# eos_cols

Plot SPRUCE


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df_pheno = df_pheno.drop(columns=["eos_2015"], errors='ignore')
# print(df_pheno)
# df_pheno = df_pheno.drop(
#     columns=[
#         "eos_2016",
#         "eos_2017",
#         "eos_2018",
#         "eos_2019", 
#         "eos_2020", 
#         "eos_2021", 
#         "eos_2022",
#         # "eos_2023",
#     ],
#     errors='ignore'
# )

temp_mapping = {
    2: "Ambient",
    4: "+4.5",
    5: "Ambient",
    6: "Control",
    7: "Ambient",
    8: "+6.75",
    9: "Ambient",
    10: "+9",
    11: "+2.25",
    13: "+4.5",
    14: "Ambient",
    15: "Ambient",
    16: "+6.75",
    17: "+9",
    19: "Control",
    20: "+2.25",
    21: "Ambient"
}

df_pheno["Temperature"] = df_pheno["plot"].map(temp_mapping)
temp_numeric_map = {
    "Control": 0,
    "+2.25": 2.25,
    "+4.5": 4.5,
    "+6.75": 6.75,
    "+9": 9
}
df_pheno["Temp_numeric"] = df_pheno["Temperature"].map(temp_numeric_map)
# print(df_pheno)

eos_cols = [col for col in df_pheno.columns if col.startswith("eos_")]
df_long = df_pheno.melt(id_vars=["plot", "Temp_numeric"], value_vars=eos_cols,
                        var_name="Year", value_name="EOS")
# print(df_long)
df_long = df_long.dropna()
# print(df_long)

groups = df_long.groupby("Temp_numeric")["EOS"].apply(list).reset_index()

data = groups["EOS"].tolist()
positions = range(len(groups))  # integer positions for each box
# print(data)
count = sum(len(inner_list) for inner_list in data)
print(count)

means = groups["EOS"].apply(np.mean)

colors = ["#8ec1da", "#cde1ec", "#ededed", "#f6d6c2", "#d47264"]

# plt.figure(figsize=(6, 3))
plt.figure(figsize=(6, 4))

bp = plt.boxplot(data, positions=positions, patch_artist=True,
                 medianprops=dict(color='black'),
                 capprops=dict(visible=False),
                 showfliers=False)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

plt.scatter(positions, means, color='black', marker='x', s=100, label='Mean', zorder=5)

for i, mean in enumerate(means):
    whisker = bp['whiskers'][2*i]  # lower whisker line for ith box
    xy = whisker.get_xydata()
    x = xy[0, 0]  # x coord of whisker line (vertical line so both points have same x)
    bottom_y = np.min(xy[:, 1])  # lowest y point of the whisker line

    plt.text(x, bottom_y - 0.5, f"{mean:.0f}", color='black',
             ha='center', va='top', fontsize=12)

tick_labels = [f"+{t}" for t in groups["Temp_numeric"]]
plt.xticks(positions, tick_labels, fontsize=12)
# plt.xticks(positions, groups["Temp_numeric"], fontsize=12)
plt.yticks(fontsize=12)

plt.xlabel("Temperature Treatment (°C)", fontsize=14)
plt.ylabel("EOS (DOY)", fontsize=14)
plt.grid(False)
plt.tight_layout()
plt.ylim(270, 340)
plt.yticks([280, 300, 320, 340])

# plt.savefig(f"../../results/figure1/SPRUCE/eos_{veg_type}.png", dpi=500, bbox_inches='tight')
os.makedirs("../../results/figure1/spruce", exist_ok=True)
plt.savefig("../../results/figure1/spruce/eos.png", dpi=500, bbox_inches='tight')
plt.show()


In [ ]:
## Show precipitation
import pandas as pd
import numpy as np

data = {
    "annual_p_2016": [803.069],
    "annual_p_2017": [883.853],
    "annual_p_2018": [871.656],
    "annual_p_2019": [885.45],
    "annual_p_2020": [600.38],
    "annual_p_2021": [616.644],
    "annual_p_2022": [851.572],
    "annual_p_2023": [699.532]
}

df = pd.DataFrame(data)

mean_p = df.mean(axis=1)[0]

std_p = df.std(axis=1)[0]

print(f"Mean annual precipitation: {mean_p:.3f} ± {std_p:.3f}")



Read PhenoCam data


In [ ]:
import pandas as pd
df = pd.read_csv('../../data/phenocam_data/tables/phenocam.csv')
# eos_cols = [c for c in df.columns if c.startswith("eos_")]
# df[eos_cols] = df[eos_cols].where(df[eos_cols] >= 200)

# valid_counts = df[eos_cols].notna().sum(axis=1)
# df = df[valid_counts >= 5].reset_index(drop=True)
df = df[df['veg_type'].isin(["EN", "DB", "DN"])]
df = df[df['latitude'] >= 30]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

def plot_eos_vs_temp(df, threshold):
    eos_cols = [c for c in df.columns if c.startswith("eos_") and c != "eos_code"]
    annual_t_cols = [c for c in df.columns if c.startswith("annual_t_")]
    annual_p_cols = [c for c in df.columns if c.startswith("annual_p_")]

    eos_years = [int(c.split("_")[1]) for c in eos_cols]
    t_map = {int(c.split("_")[2]): c for c in annual_t_cols}
    p_map = {int(c.split("_")[2]): c for c in annual_p_cols}

    common_years = [y for y in eos_years if y in t_map and y in p_map]
    c_eos = [f"eos_{y}" for y in common_years]
    c_t = [t_map[y] for y in common_years]
    c_p = [p_map[y] for y in common_years]

    eos_mat = df[c_eos].to_numpy(dtype=float)
    t_mat = df[c_t].to_numpy(dtype=float)
    p_mat = df[c_p].to_numpy(dtype=float)

    t_mat[np.isnan(eos_mat)] = np.nan
    p_mat[np.isnan(eos_mat)] = np.nan

    with np.errstate(all="ignore"):
        mean_eos = np.nanmean(eos_mat, axis=1)
        mean_t = np.nanmean(t_mat, axis=1)
        mean_p = np.nanmean(p_mat, axis=1)

    df_out = df.copy()
    df_out["longterm_eos"] = mean_eos
    df_out["longterm_annual_t"] = mean_t
    if np.nanmean(df_out["longterm_annual_t"]) > 100:
        df_out["longterm_annual_t"] = df_out["longterm_annual_t"] - 273.15  # convert K → °C
    df_out["longterm_annual_p"] = mean_p
    df_out = df_out[(df_out["longterm_eos"] > 200) & np.isfinite(df_out["longterm_annual_t"]) & np.isfinite(df_out["longterm_annual_p"])].copy()

    red_mask = df_out["longterm_annual_p"] <= threshold
    gray_mask = ~red_mask
    dry_color = "#e03c31"  # red tone replacement

    def quadratic_regression(x, y):
        X = pd.DataFrame({"x": x, "x2": x**2})
        X = sm.add_constant(X)
        return sm.OLS(y, X, missing="drop").fit()

    red_model = quadratic_regression(df_out.loc[red_mask, "longterm_annual_t"],
                                     df_out.loc[red_mask, "longterm_eos"])

    def format_p(p):
        if p < 0.01:
            return "< 0.01"
        elif p < 0.05:
            return "< 0.05"
        else:
            return f"= {p:.2f}"

    fig = plt.figure(figsize=(6, 4))

    plt.scatter(df_out.loc[gray_mask, "longterm_annual_t"],
                df_out.loc[gray_mask, "longterm_eos"],
                color="gray", s=35, alpha=0.5, edgecolor="none",
                label="Wet sites")
    plt.scatter(df_out.loc[red_mask, "longterm_annual_t"],
                df_out.loc[red_mask, "longterm_eos"],
                color=dry_color, s=35, alpha=0.5, edgecolor="none",
                label="Dry sites")

    x_group = df_out.loc[red_mask, "longterm_annual_t"].dropna()
    if len(x_group) > 1:
        x_vals = np.linspace(x_group.min(), x_group.max(), 200)
        X_pred = pd.DataFrame({"x": x_vals, "x2": x_vals**2})
        X_pred = sm.add_constant(X_pred)

        pred = red_model.get_prediction(X_pred)
        pred_summary = pred.summary_frame(alpha=0.05)

        y_pred = pred_summary["mean"]
        ci_lower = pred_summary["mean_ci_lower"]
        ci_upper = pred_summary["mean_ci_upper"]

        plt.plot(x_vals, y_pred, color=dry_color, linewidth=2, label="Quadratic fit (dry)")
        plt.fill_between(x_vals, ci_lower, ci_upper, color=dry_color, alpha=0.15, label="95% CI")

    plt.text(0.03, 0.97,
             f"R² = {red_model.rsquared:.2f}\np(t²) {format_p(red_model.pvalues['x2'])}\np(t)  {format_p(red_model.pvalues['x'])}",
             color=dry_color, transform=plt.gca().transAxes,
             fontsize=12, va="top")

    plt.xlabel("MAT (°C)", fontsize=14)
    plt.ylabel("EOS (DOY)", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.tight_layout()
    return fig, df_out


Plot PhenoCam


In [ ]:
threshold  = 0.8
fig, long_term_mean_phenocam = plot_eos_vs_temp(df, threshold)
os.makedirs("../../results/figure1/phenocam", exist_ok=True)
fig.savefig("../../results/figure1/phenocam/eos.png", dpi=500, bbox_inches='tight')

In [ ]:
long_term_mean_phenocam
df = long_term_mean_phenocam.dropna(axis=1, how='all')
df.drop(columns='longterm_eos', inplace=True)
counts = df.filter(like='eos').count()
total_eos = counts.sum()
print(total_eos)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm

def compare_linear_quadratic(df_out, threshold, dataset_name):
        import statsmodels.api as sm
    from statsmodels.stats.anova import anova_lm

    dry = df_out[df_out["longterm_annual_p"] <= threshold].copy()
    x = dry["longterm_annual_t"].astype(float)
    y = dry["longterm_eos"].astype(float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = len(x)

    X_lin = sm.add_constant(pd.DataFrame({"t": x}))
    X_quad = sm.add_constant(pd.DataFrame({"t": x, "t2": x**2}))
    m_lin = sm.OLS(y, X_lin).fit()
    m_quad = sm.OLS(y, X_quad).fit()
    nested = anova_lm(m_lin, m_quad)

    print("=" * 72)
    print(f"{dataset_name}: linear vs quadratic (dry sites, MAP <= {threshold} m/day)")
    print(f"  n = {n}")
    print("-" * 72)
    print(f"{'Model':<12} {'R²':>8} {'Adj R²':>8} {'AIC':>10} {'BIC':>10}")
    print(f"{'Linear':<12} {m_lin.rsquared:8.2f} {m_lin.rsquared_adj:8.2f} {m_lin.aic:10.2f} {m_lin.bic:10.2f}")
    print(f"{'Quadratic':<12} {m_quad.rsquared:8.2f} {m_quad.rsquared_adj:8.2f} {m_quad.aic:10.2f} {m_quad.bic:10.2f}")
    print("-" * 72)
    print("Linear coefficients:")
    print(f"  intercept = {m_lin.params['const']:.3f}  (p={m_lin.pvalues['const']:.4g})")
    print(f"  t         = {m_lin.params['t']:.3f}  (p={m_lin.pvalues['t']:.4g})")
    print("Quadratic coefficients:")
    print(f"  intercept = {m_quad.params['const']:.3f}  (p={m_quad.pvalues['const']:.4g})")
    print(f"  t         = {m_quad.params['t']:.3f}  (p={m_quad.pvalues['t']:.4g})")
    print(f"  t²        = {m_quad.params['t2']:.3f}  (p={m_quad.pvalues['t2']:.4g})")
    print("-" * 72)
    f_stat = nested.loc[1, "F"]
    p_nested = nested.loc[1, "Pr(>F)"]
    dAIC = m_quad.aic - m_lin.aic
    prefer = "quadratic" if (p_nested < 0.05 and dAIC < 0) else (
        "quadratic (AIC)" if dAIC < 0 else (
            "quadratic (F-test)" if p_nested < 0.05 else "linear"
        )
    )
    print(f"Nested F-test (quad vs lin): F = {f_stat:.3f}, p = {p_nested:.4g}")
    print(f"ΔAIC (quad − lin) = {dAIC:.2f}  → prefer: {prefer}")
    print("=" * 72)
    return m_lin, m_quad, nested

_ = compare_linear_quadratic(long_term_mean_phenocam, threshold=0.8, dataset_name="PhenoCam")


Read FLUXNET data


In [ ]:
import pandas as pd
df = pd.read_csv('../../data/flux_data/tables/flux_50.csv')
df = df[df['latitude'] >= 30]
eos_cols = [c for c in df.columns if c.startswith("eos_")]
# df[eos_cols] = df[eos_cols].where(df[eos_cols] >= 200)
# # df = df[df['veg_type']=='DBF']
# valid_counts = df[eos_cols].notna().sum(axis=1)
# df = df[valid_counts >= 5].reset_index(drop=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

def plot_eos_vs_temp(df, threshold):
    eos_cols = [c for c in df.columns if c.startswith("eos_") and c != "eos_code"]
    annual_t_cols = [c for c in df.columns if c.startswith("annual_t_")]
    annual_p_cols = [c for c in df.columns if c.startswith("annual_p_")]

    eos_years = [int(c.split("_")[1]) for c in eos_cols]
    t_map = {int(c.split("_")[2]): c for c in annual_t_cols}
    p_map = {int(c.split("_")[2]): c for c in annual_p_cols}

    common_years = [y for y in eos_years if y in t_map and y in p_map]
    c_eos = [f"eos_{y}" for y in common_years]
    c_t = [t_map[y] for y in common_years]
    c_p = [p_map[y] for y in common_years]

    eos_mat = df[c_eos].to_numpy(dtype=float)
    t_mat = df[c_t].to_numpy(dtype=float)
    p_mat = df[c_p].to_numpy(dtype=float)

    t_mat[np.isnan(eos_mat)] = np.nan
    p_mat[np.isnan(eos_mat)] = np.nan

    with np.errstate(all="ignore"):
        mean_eos = np.nanmean(eos_mat, axis=1)
        mean_t = np.nanmean(t_mat, axis=1)
        mean_p = np.nanmean(p_mat, axis=1)

    df_out = df.copy()
    df_out["longterm_eos"] = mean_eos
    df_out["longterm_annual_t"] = mean_t
    if np.nanmean(df_out["longterm_annual_t"]) > 100:
        df_out["longterm_annual_t"] = df_out["longterm_annual_t"] - 273.15  # convert K → °C
    df_out["longterm_annual_p"] = mean_p
    df_out = df_out[(df_out["longterm_eos"] > 200) & np.isfinite(df_out["longterm_annual_t"]) & np.isfinite(df_out["longterm_annual_p"])].copy()

    red_mask = df_out["longterm_annual_p"] <= threshold
    gray_mask = ~red_mask
    dry_color = "#e03c31"  # red tone replacement

    def quadratic_regression(x, y):
        X = pd.DataFrame({"x": x, "x2": x**2})
        X = sm.add_constant(X)
        return sm.OLS(y, X, missing="drop").fit()

    red_model = quadratic_regression(df_out.loc[red_mask, "longterm_annual_t"],
                                     df_out.loc[red_mask, "longterm_eos"])

    def format_p(p):
        if p < 0.01:
            return "< 0.01"
        elif p < 0.05:
            return "< 0.05"
        else:
            return f"= {p:.2f}"

    fig = plt.figure(figsize=(6, 4))

    plt.scatter(df_out.loc[gray_mask, "longterm_annual_t"],
                df_out.loc[gray_mask, "longterm_eos"],
                color="gray", s=35, alpha=0.5, edgecolor="none",
                label="Wet sites")
    plt.scatter(df_out.loc[red_mask, "longterm_annual_t"],
                df_out.loc[red_mask, "longterm_eos"],
                color=dry_color, s=35, alpha=0.5, edgecolor="none",
                label="Dry sites")

    x_group = df_out.loc[red_mask, "longterm_annual_t"].dropna()
    if len(x_group) > 1:
        x_vals = np.linspace(x_group.min(), x_group.max(), 200)
        X_pred = pd.DataFrame({"x": x_vals, "x2": x_vals**2})
        X_pred = sm.add_constant(X_pred)

        pred = red_model.get_prediction(X_pred)
        pred_summary = pred.summary_frame(alpha=0.05)

        y_pred = pred_summary["mean"]
        ci_lower = pred_summary["mean_ci_lower"]
        ci_upper = pred_summary["mean_ci_upper"]

        plt.plot(x_vals, y_pred, color=dry_color, linewidth=2, label="Quadratic fit (dry)")
        plt.fill_between(x_vals, ci_lower, ci_upper, color=dry_color, alpha=0.15, label="95% CI")

    plt.text(0.03, 0.97,
             f"R² = {red_model.rsquared:.2f}\np(t²) {format_p(red_model.pvalues['x2'])}\np(t)  {format_p(red_model.pvalues['x'])}",
             color=dry_color, transform=plt.gca().transAxes,
             fontsize=12, va="top")

    plt.xlabel("MAT (°C)", fontsize=14)
    plt.ylabel("EOS (DOY)", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.tight_layout()
    return fig, df_out


Plot FLUXNET


In [ ]:
threshold  = 1.0
fig, long_term_mean_flux = plot_eos_vs_temp(df, threshold)
os.makedirs("../../results/figure1/flux", exist_ok=True)
fig.savefig("../../results/figure1/flux/eos.png", dpi=500, bbox_inches='tight')


In [ ]:
long_term_mean_flux
df = long_term_mean_flux.dropna(axis=1, how='all')
df.drop(columns='longterm_eos', inplace=True)
counts = df.filter(like='eos').count()
total_eos = counts.sum()
print(total_eos)

In [ ]:
_ = compare_linear_quadratic(long_term_mean_flux, threshold=1.0, dataset_name="FLUXNET")


Read PEP725 data


In [ ]:
import os
import pandas as pd
df = pd.read_csv("../../data/pep725_data/tables/pep725_revised_06052026.csv")


Plot PEP725


### PEP725 all species


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

df_pep_raw = pd.read_csv("../../data/pep725_data/tables/pep725_revised_06052026.csv")
#   Betula   → Betula pendula (silver birch)
#   Quercus  → Quercus robur (pedunculate oak)
#   Fagus    → Fagus sylvatica (European beech)  [excluded from figure]
df_pep_raw = df_pep_raw[df_pep_raw["species"] != "Fagus"].copy()
print("Species in figure:", sorted(df_pep_raw["species"].unique()))

eos_cols = [c for c in df_pep_raw.columns if c.startswith("eos_") and c != "eos_code"]
temp_cols = [c for c in df_pep_raw.columns if c.startswith("annual_t_")]
precip_cols = [c for c in df_pep_raw.columns if c.startswith("annual_p_")]

df_eos = df_pep_raw.melt(id_vars=["PEP_ID", "species", "latitude"], value_vars=eos_cols, var_name="Year", value_name="EOS")
df_eos["Year"] = df_eos["Year"].str.replace("eos_", "").astype(int)

df_temp = df_pep_raw.melt(id_vars=["PEP_ID", "species"], value_vars=temp_cols, var_name="Year", value_name="Annual_Temp")
df_temp["Year"] = df_temp["Year"].str.replace("annual_t_", "").astype(int)

df_precip = df_pep_raw.melt(id_vars=["PEP_ID", "species"], value_vars=precip_cols, var_name="Year", value_name="Annual_Precip")
df_precip["Year"] = df_precip["Year"].str.replace("annual_p_", "").astype(int)

merged = pd.merge(df_eos, df_temp, on=["PEP_ID", "species", "Year"])
merged = pd.merge(merged, df_precip, on=["PEP_ID", "species", "Year"])
merged = merged.dropna(subset=["EOS", "Annual_Temp", "Annual_Precip"])

if merged["Annual_Temp"].mean() > 200:
    merged["Annual_Temp"] = merged["Annual_Temp"] - 273.15

TEMP_MIN, TEMP_MAX = 5.0, 12.0

clean_open_df = (
    merged
    .groupby(["PEP_ID", "species"])
    .agg({"EOS": "mean", "Annual_Temp": "mean", "Annual_Precip": "mean"})
    .reset_index()
    .rename(columns={"EOS": "mean_EOS", "Annual_Temp": "mean_Temp", "Annual_Precip": "mean_Precip"})
)

clean_open_df = clean_open_df[(clean_open_df["mean_Temp"] >= TEMP_MIN) & (clean_open_df["mean_Temp"] <= TEMP_MAX)].copy()

bin_edges = np.arange(TEMP_MIN, TEMP_MAX + 0.5, 0.5)
bin_labels = [f"{bin_edges[i]:.1f}" for i in range(len(bin_edges) - 1)]
clean_open_df["Temp_Bin"] = pd.cut(clean_open_df["mean_Temp"], bins=bin_edges, labels=bin_labels, include_lowest=True)
clean_open_df = clean_open_df.dropna(subset=["Temp_Bin"])

P_THRESHOLD = 1.0
low_label = "Dry regions"
high_label = "Wet regions"
clean_open_df["Precip_Group"] = np.where(clean_open_df["mean_Precip"] <= P_THRESHOLD, low_label, high_label)

chunks_with_p = []
for _, group_frame in clean_open_df.groupby(["species", "Precip_Group", "Temp_Bin"], observed=False):
    chunks_with_p.append(group_frame if len(group_frame) >= 10 else group_frame.iloc[0:0])
trimmed_with_p = pd.concat(chunks_with_p, ignore_index=True)

chunks_all_p = []
for _, group_frame in clean_open_df.groupby(["Precip_Group", "Temp_Bin"], observed=False):
    chunks_all_p.append(group_frame if len(group_frame) >= 10 else group_frame.iloc[0:0])
trimmed_all_p = pd.concat(chunks_all_p, ignore_index=True)

fig_keys = trimmed_all_p[["PEP_ID", "species"]].drop_duplicates()
site_years_fig = merged.merge(fig_keys, on=["PEP_ID", "species"], how="inner")
n_sites = trimmed_all_p["PEP_ID"].nunique()
n_site_species = len(trimmed_all_p)
n_site_years = len(site_years_fig)
print("PEP725 all-species figure sample sizes")
print(f"  Sites (unique PEP_ID): {n_sites}")
print(f"  Site–species means plotted: {n_site_species}")
print(f"  Site–year observations (underlying): {n_site_years}")
for grp in [low_label, high_label]:
    sub = trimmed_all_p[trimmed_all_p["Precip_Group"] == grp]
    keys = sub[["PEP_ID", "species"]].drop_duplicates()
    sy = merged.merge(keys, on=["PEP_ID", "species"], how="inner")
    print(f"  {grp}: sites={sub['PEP_ID'].nunique()}, site–species={len(sub)}, site–years={len(sy)}")

palette = {low_label: "#e03c31", high_label: "gray"}
hue_order = [low_label, high_label]

def plot_precip_split_panel(ax, data, ylabel="EOS (DOY)", show_legend=False, box_width=0.6, hue_order=None, palette_colors=None):
    if data.empty:
        return
    hue_order = hue_order if hue_order is not None else [low_label, high_label]
    palette_colors = palette_colors if palette_colors is not None else palette

    sns.boxplot(
        data=data, x="Temp_Bin", y="mean_EOS", hue="Precip_Group", hue_order=hue_order,
        palette=palette_colors, width=box_width, linewidth=0.25, showfliers=False, showcaps=False,
        boxprops=dict(linewidth=0.25),
        whiskerprops=dict(linewidth=0.6, alpha=0.3, color="0.35"),
        medianprops=dict(linewidth=0.4, alpha=0.4),
        ax=ax, dodge=True, zorder=2
    )
    sns.pointplot(
        data=data, x="Temp_Bin", y="mean_EOS", hue="Precip_Group", hue_order=hue_order,
        palette=palette_colors, estimator=np.median, errorbar=None, dodge=0.4,
        markers="none", linestyles="-", linewidth=1.0, ax=ax, zorder=4
    )
    ax.set_ylabel(ylabel, fontsize=14)
    ax.tick_params(axis="y", labelsize=11)
    ax.grid(False)
    if show_legend:
        legend_elements = [
            Patch(facecolor="#e03c31", edgecolor="none", label="Dry sites"),
            Patch(facecolor="gray", edgecolor="none", label="Wet sites"),
        ]
        ax.legend(handles=legend_elements, loc="upper left", frameon=False, fontsize=12)
    elif ax.get_legend() is not None:
        ax.get_legend().remove()

sns.set_theme(style="ticks")
fig_all, ax_all = plt.subplots(figsize=(5, 4))
plot_precip_split_panel(ax_all, trimmed_all_p, ylabel="EOS (DOY)", show_legend=True, hue_order=hue_order, palette_colors=palette)
ax_all.set_xlabel("MAT (°C)", fontsize=14)
xtick_labels = []
for lab in ax_all.get_xticklabels():
    try:
        v = float(lab.get_text())
        xtick_labels.append(f"{v:.0f}" if abs(v - round(v)) < 1e-6 else "")
    except ValueError:
        xtick_labels.append("")
ax_all.set_xticklabels(xtick_labels, fontsize=12)
for tick, lab in zip(ax_all.xaxis.get_major_ticks(), xtick_labels):
    vis = lab != ""
    tick.tick1line.set_visible(vis)
    tick.tick2line.set_visible(False)  # no top x-ticks
for spine in ax_all.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.6)
ax_all.tick_params(width=0.6, top=False, right=False)
plt.tight_layout()
os.makedirs("../../results/figure1/pep725", exist_ok=True)
fig_all.savefig("../../results/figure1/pep725/pep725_all_species_p1.0.png", dpi=500, bbox_inches="tight")
plt.show()



### PEP725 by species


In [ ]:
species_list = trimmed_with_p["species"].unique()
os.makedirs("../../results/figure1/pep725", exist_ok=True)

for species_name in species_list:
    sp_with_p = trimmed_with_p[trimmed_with_p["species"] == species_name]
    fig_sp, ax_sp = plt.subplots(figsize=(5, 4))
    plot_precip_split_panel(ax_sp, sp_with_p, ylabel="EOS (DOY)", show_legend=True, hue_order=hue_order, palette_colors=palette)
    ax_sp.set_xlabel("MAT (°C)", fontsize=14)

    xtick_labels = []
    for lab in ax_sp.get_xticklabels():
        try:
            v = float(lab.get_text())
            xtick_labels.append(f"{v:.0f}" if abs(v - round(v)) < 1e-6 else "")
        except ValueError:
            xtick_labels.append("")
    ax_sp.set_xticklabels(xtick_labels, fontsize=12)
    for tick, lab in zip(ax_sp.xaxis.get_major_ticks(), xtick_labels):
        vis = lab != ""
        tick.tick1line.set_visible(vis)
        tick.tick2line.set_visible(False)  # no top x-ticks
    for spine in ax_sp.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.6)
    ax_sp.tick_params(width=0.6, top=False, right=False)

    plt.tight_layout()
    out_fp = f"../../results/figure1/pep725/pep725_{species_name.lower()}_p1.0.png"
    fig_sp.savefig(out_fp, dpi=500, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_fp}")

